In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
data=pd.read_csv(r"C:\Users\ozgur\OneDrive\Desktop\phyton\veri_setleri\spain gender violance\Gender Violence Spain Dataset (2020-2024).csv")
df=data.copy()
display(df.head())

In [ ]:
# import os
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# import seaborn as sns
# from scipy import stats
# pd.set_option('display.float_format', lambda x: '%.2f' % x)
# file_name = "Gender Violence Spain Dataset (2020-2024).csv"
# kaggle_path = f"/kaggle/input/spain-gender-violence/{file_name}"
# if os.path.exists(kaggle_path):
#     df = pd.read_csv(kaggle_path)
# elif os.path.exists(file_name):
#     df = pd.read_csv(file_name)
# else:
#     df = pd.read_csv(r"C:\Users\ozgur\OneDrive\Desktop\phyton\veri_setleri\spain gender violance\Gender Violence Spain Dataset (2020-2024).csv")
# display(df.head())

In [ ]:
df["Funding_EUR"] = pd.to_numeric(
    df["Funding_EUR"].astype(str).str.replace("€", "").str.replace(",", "").str.strip(),
    errors='coerce'
)
print(f"Eksik Funding_EUR Gözlem Sayısı: {df['Funding_EUR'].isna().sum()}")
display(df[df["Funding_EUR"].isna()])

**Sadece Navarra ve País Vasco bölgelerine ait veriler olmadığı için bu iki bölge veri setinden silinir.**

In [ ]:
df["Vi_Co_ratio"]=df["Violence_Victims"]/df["Complaints"] #Violance/Complaints ratio
df["Co_Vi_ratio"]=df["Complaints"]/df["Violence_Victims"] #Complaints/Violance ratio
df_z=df.dropna(subset=["Funding_EUR"]).copy()
display(df_z["Funding_EUR"].isnull().values.sum())

$H_0$ (Yokluk Hipotezi / Null Hypothesis): Devletin ayırdığı fon miktarı (Funding_EUR) ile şikayet başına düşen şiddet mağduru oranı (Vi_Co_ratio) arasında istatistiksel olarak anlamlı bir ilişki (korelasyon) yoktur ($r = 0$ veya $\rho = 0$).

$H_1$ (Alternatif Hipotez / Alternative Hypothesis): Devletin ayırdığı fon miktarı (Funding_EUR) ile şikayet başına düşen şiddet mağduru oranı (Vi_Co_ratio) arasında istatistiksel olarak anlamlı bir ilişki vardır ($r \neq 0$ veya $\rho \neq 0$).

In [ ]:
df_numeric=df_z.select_dtypes(include=["int64","float64"])
corr_matrix=df_numeric.corr()
sns.heatmap(data=corr_matrix,fmt=".2g",cmap="coolwarm",annot=True,linewidths=0.5)
plt.title("Heatmap")
plt.show()

In [ ]:
sns.pairplot(data=df_z,kind="reg");

In [ ]:
df_numeric=df_z.select_dtypes(include=["int64","float64"])
for i in df_numeric:
    plt.figure(figsize=(8,2))
    sns.boxplot(x=df_numeric[i]);
    plt.title(f"{i} boxplot")
    plt.show()

In [ ]:
for i in df_numeric:
    Q1=df_z[i].quantile(0.25)
    Q3=df_z[i].quantile(0.75)
    IQR=Q3-Q1
    lower=Q1-1.5*IQR
    upper=Q3+1.5*IQR
    outliars=df_z[(df_z[i]<lower )|( df_z[i]>upper)]
    print(f"{i} için aykırı değerler")
    display(len(outliars))

In [ ]:
from scipy.stats import shapiro
values=df_z[["Vi_Co_ratio","Funding_EUR"]]
for col in values.columns:
    test_ist,p_value=shapiro(values[col])
    if p_value<0.05:
        print(f"{col} normal dağılmaz.\np-value:{p_value:.4f}")
    else:
        print(f"{col} normal dağılır.np-value:{p_value:.4f}")

In [ ]:
sns.histplot(data=df_z["Funding_EUR"])
plt.title("Funding_EUR Histogram")
plt.show()

In [ ]:
df_z["Log_Funding_EUR"]=np.log(df_z["Funding_EUR"])
test_ist,p_value=shapiro(df_z["Log_Funding_EUR"])
if p_value<0.05:
    print(f"normal dağılmaz.\n" \
    f"test ist: {test_ist:.4f}\n"
    f"p value: {p_value:.4f}")
else:
    print("Log_Funding_EUR normal dağılır\n"
    f"test ist: {test_ist:.4f}\n" \
    f"p value: {p_value:.4f}")

In [ ]:
sns.histplot(data=df_z["Log_Funding_EUR"])
plt.title("Log_Funding_EUR Histohram")
plt.show()
df_z["Log_Funding_EUR"].describe().T

In [ ]:
sns.boxplot(data=df_z["Funding_EUR"]);
df_z.sort_values(by="Funding_EUR",ascending=False).head()

**Cataluña verisinin 2021 Funding_EUR değeri veri girişi hatasıdır başında bir adet 1 hatalı girilmiştir.**

In [ ]:
df_z.loc[(df_z["Region"] == "Cataluña") & (df_z["Year"] == 2021), "Funding_EUR"] = 11666598.59
df_z["Log_Funding_EUR"] = np.log(df_z["Funding_EUR"])
test_ist, p_value = shapiro(df_z["Log_Funding_EUR"])
if p_value < 0.05:
    print(f"Log_Funding_EUR normal dağılmaz.\nTest İstatistiği (W): {test_ist:.4f}\np-değeri: {p_value:.4f}")
else:
    print(f"Log_Funding_EUR normal dağılır.\nTest İstatistiği (W): {test_ist:.4f}\np-değeri: {p_value:.4f}")

In [ ]:
sns.histplot(data=df_z["Log_Funding_EUR"], kde=True)
plt.title("Düzeltilmiş Log_Funding_EUR Histogramı")
plt.show()

In [ ]:
sns.boxplot(x=df_z["Vi_Co_ratio"]);

In [ ]:
sns.histplot(data=df_z["Vi_Co_ratio"]);
df_z["Vi_Co_ratio"].describe().T

In [ ]:
df_z["Log_Vi_Co_ratio"]=np.log(df_z["Vi_Co_ratio"])
sns.histplot(data=df_z["Log_Vi_Co_ratio"]);

In [ ]:
test_ist,p_value=shapiro(df_z["Log_Vi_Co_ratio"])
if p_value<0.05:
    print("normal dağılmaz.")
else:
    print("Log_Vi_Co_ratio normal dağılır\n"
    f"test ist: {test_ist:.4f}\n" \
    f"p value: {p_value:.4f}")

In [ ]:
sns.scatterplot(data=df_z,x="Log_Funding_EUR",y="Log_Vi_Co_ratio");

In [ ]:
from scipy.stats import spearmanr
rho_degeri, p_value = spearmanr(df_z["Funding_EUR"], df_z["Vi_Co_ratio"])
print(f"Spearman Korelasyon Katsayısı (rho): {rho_degeri:.4f}\n"
      f"p-değeri (p-value): {p_value:.4f}")
if p_value < 0.05:
    print(f"Karar: H0 reddedilir. Aralarında istatistiksel olarak anlamlı ve negatif bir ilişki vardır (rho = {rho_degeri:.2f}).")
else:
    print(f"Karar: H0 reddedilemez. Aralarında anlamlı bir ilişki yoktur (rho = {rho_degeri:.2f}).")

### **Sonuç** : 
Devletin ayırdığı fon miktarı (Funding_EUR) ile şikayet başına düşen şiddet mağduru oranı (Vi_Co_ratio) arasında istatistiksel olarak anlamlı, negatif yönlü ve orta güçte bir ilişki bulunmaktadır ($\rho = -0.55, p < 0.001$). Fon miktarının daha yüksek olduğu bölgelerde, yapılan şikayetlere kıyasla kesinleşmiş mağdur oranı daha düşük seyretmektedir.

### **Bölgeler Arasındaki Şiddet/Şikayet Oranları Arasında Fark Var Mı?** 

$H_0$ (Sıfır Hipotezi - Null Hypothesis): Tüm bölgelerin ortalama Vi_Co_ratio değerleri eşittir ($\mu_1 = \mu_2 = \dots = \mu_k$). Yani, bölgeler arasında ortalama Vi_Co_ratio açısından anlamlı bir fark yoktur.

$H_1$ (Alternatif Hipotez - Alternative Hypothesis): En az bir bölgenin ortalama Vi_Co_ratio değeri diğerlerinden farklıdır.

**Step 1: Normality test**

In [ ]:
sns.histplot(data=df["Vi_Co_ratio"])
plt.title("Vi_Co_ratio Histogram")
plt.show()
sns.boxplot(data=df["Vi_Co_ratio"])
plt.show()
display(df["Vi_Co_ratio"].describe())

In [ ]:
from scipy.stats import shapiro
w_test,p_value=shapiro(df["Vi_Co_ratio"])
print(f"P value:{p_value:.4f}\n" \
f"Test ist.:{w_test:.4f}")
if p_value<0.05:
    print("H0 red edilir.\n" \
    "Veri normal dağılmaz.")
else:
    print("H0 red edilemez.\n" \
    "Veri normal dağılır.")

In [ ]:
Q1=df["Vi_Co_ratio"].quantile(0.25)
Q3=df["Vi_Co_ratio"].quantile(0.75)
IQR=Q3-Q1
lower=Q1-1.5*IQR
upper=Q3+1.5*IQR
display(df[df["Vi_Co_ratio"]>upper])
print(f"upper değeri:{upper:.4f}")

In [ ]:
# Log transformation
df["Log_Vi_Co_ratio"]=np.log(df["Vi_Co_ratio"])
sns.histplot(data=df,x="Log_Vi_Co_ratio")

In [ ]:
from scipy.stats import shapiro
w_test,p_value=shapiro(df["Log_Vi_Co_ratio"])
print(f"P value:{p_value:.4f}\n" \
f"Test ist.:{w_test:.4f}")
if p_value<0.05:
    print("H0 red edilir.\n" \
    "Veri normal dağılmaz.")
else:
    print("H0 red edilemez.\n" \
    "Veri normal dağılır.")

**Step 2: Equal Variance Test**

In [ ]:
from scipy.stats import levene
groups = [df[df["Region"] == region]["Log_Vi_Co_ratio"] for region in df["Region"].unique()]
# Yıldız (*) operatörü (unpacking) ile listedeki her bir grubu levene fonksiyonuna ayrı birer argüman olarak gönderiyoruz.
w_test, p_value = levene(*groups)
print(f"W_test: {w_test:.4f}")
print(f"P value: {p_value:.4f}")
if p_value < 0.05:
    print("H0 reddedilir. Varyans homojenliği sağlanmaz.")
else:
    print("H0 reddedilemez. Varyans homojenliği sağlanır.")


In [ ]:
import pingouin as pg
welch=pg.welch_anova(data=df,dv="Log_Vi_Co_ratio",between="Region")
welch

**Sonuç: H0 red edilir. F=88.54 ve P value<0.001 tir. Bu yüzden bölgeler arasında anlamlı fark vardır.**
**Farkı anlamak için Games-Howel Post-Hoc testi yapılır.**

In [ ]:
# Anlamlı fark bulunanlar.
games_howel=pg.pairwise_gameshowell(data=df,dv="Log_Vi_Co_ratio",between="Region")
display(games_howel[games_howel["pval"]<0.05].sort_values(by="hedges",key=abs,ascending=False).reset_index())
games_howel[games_howel["pval"]<0.05].shape

In [ ]:
corr_matrix=df_numeric.corr()
sns.heatmap(data=corr_matrix,fmt=".2g",cmap="coolwarm",annot=True,linewidths=0.5)
plt.title("Heatmap")
plt.show()

In [ ]:
from scipy import stats
stat_fund, p_fund = stats.shapiro(df_z["Funding_EUR"])
stat_ratio, p_ratio = stats.shapiro(df_z["Vi_Co_ratio"])

print(f"Shapiro-Wilk Normallik Testi:")
print(f"Funding_EUR p-değeri: {p_fund:.4e} (p < 0.05, Normal dağılmıyor)")
print(f"Vi_Co_ratio p-değeri: {p_ratio:.4e} (p < 0.05, Normal dağılmıyor)\n")

spearman_r, spearman_p = stats.spearmanr(df_z["Funding_EUR"], df_z["Vi_Co_ratio"])
pearson_r, pearson_p = stats.pearsonr(df_z["Funding_EUR"], df_z["Vi_Co_ratio"])

print(f"Korelasyon Analizi Sonuçları:")
print(f"Spearman (rho): {spearman_r:.4f}, p-değeri: {spearman_p:.4f}")
print(f"Pearson (r)   : {pearson_r:.4f}, p-değeri: {pearson_p:.4f}")

**Sonuç ve Değerlendirme:**
* **Varsayım Kontrolü:** Shapiro-Wilk testi sonuçlarına göre her iki değişken için de $p < 0.05$ olduğundan normallik varsayımı sağlanmamaktadır. Bu nedenle parametrik olmayan **Spearman Sıra Korelasyonu (Spearman's Rank Correlation)** tercih edilmiştir.
* **Hipotez Testi Sonucu:** Hesaplanan p-değeri $\alpha = 0.05$ anlamlılık düzeyinden büyük olduğu için **$H_0$ (Yokluk Hipotezi) reddedilemez**. Devletin ayırdığı fon miktarı ile şikayet başına düşen şiddet mağduru oranı arasında istatistiksel olarak anlamlı bir ilişki bulunmamıştır.

# **Regresyon**

In [ ]:
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan
from scipy.stats import shapiro

X = df_z[["Log_Funding_EUR", "School_Leaving(%)", "Unemployment_Rate(%)"]]
X = sm.add_constant(X)
y = df_z["Log_Vi_Co_ratio"]

model = sm.OLS(y, X).fit()
print(model.summary())

residuals = model.resid
fitted = model.fittedvalues

# a) Artıkların Normalliği
w_stat, p_shapiro = shapiro(residuals)
print(f"\nShapiro-Wilk (Artıklar) -> W: {w_stat:.4f}, p: {p_shapiro:.4f}")
if p_shapiro < 0.05:
    print("Karar: H0 reddedilir, artıklar normal dağılmaz.")
else:
    print("Karar: H0 reddedilemez, artıklar normal dağılır.")

# b) Eşvaryanslılık
lm_stat, p_bp, _, _ = het_breuschpagan(residuals, model.model.exog)
print(f"\nBreusch-Pagan (Eşvaryanslılık) -> LM: {lm_stat:.4f}, p: {p_bp:.4f}")
if p_bp < 0.05:
    print("Karar: H0 reddedilir, değişen varyans (heteroscedasticity) vardır.")
else:
    print("Karar: H0 reddedilemez, sabit varyans (homoscedasticity) sağlanır.")

# c) Çoklu Doğrusallık(VIF)
vif_df = pd.DataFrame({
    "Değişken": X.columns,
    "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
})
print("\nVIF Değerleri:\n", vif_df[vif_df["Değişken"] != "const"])

### **Regresyon Analizi Sonuç ve Değerlendirme**

* **Modelin Genel Anlamlılığı (Overall Significance):** Kurulan Çoklu Doğrusal Regresyon (Multiple Linear Regression - OLS) modeli bir bütün olarak istatistiksel olarak anlamlıdır ($F = 15.42, p < 0.001$).
* **Açıklayıcılık Gücü (Goodness of Fit):** Belirlilik Katsayısı $R^2 = 0.363$ (Düzeltilmiş / Adjusted $R^2 = 0.340$) bulunmuştur. Bağımsız değişkenler, şikayet başına düşen mağdur oranının logaritmasındaki (`Log_Vi_Co_ratio`) değişimin yaklaşık **%36.3'ünü** açıklamaktadır.
* **Değişkenlerin Etkisi (Coefficients):**
  * **`Log_Funding_EUR` ($\beta = -0.2346, p < 0.001$):** İstatistiksel olarak anlamlı ve negatif yönlü bir etkiye sahiptir. Her iki değişken de logaritmik (Log-Log model / esneklik) olduğundan, diğer değişkenler sabitken devlet fonlarındaki **%1'lik bir artış**, şikayet başına düşen mağdur oranında ortalama **%0.23'lük bir azalış** ile ilişkilidir.
  * **`School_Leaving(%)` ($\beta = 0.0039, p = 0.661$) ve `Unemployment_Rate(%)` ($\beta = 0.0015, p = 0.835$):** $p > 0.05$ olduğu için bu değişkenlerin model üzerinde istatistiksel olarak anlamlı bir etkisi bulunmamıştır.
* **Varsayım Kontrolleri (Assumption Checks):**
  * **Artıkların Normalliği (Normality of Residuals):** Shapiro-Wilk testi ($W = 0.9893, p = 0.7165 > 0.05$) artıkların normal dağıldığını doğrulamaktadır.
  * **Eşvaryanslılık (Homoscedasticity):** Breusch-Pagan testi ($LM = 2.0185, p = 0.5686 > 0.05$) sabit varyans varsayımının sağlandığını göstermektedir.
  * **Çoklu Doğrusallık (Multicollinearity):** Tüm bağımsız değişkenler için $\text{VIF} < 1.74$ olduğundan çoklu doğrusallık problemi yoktur.
  * **Otokorelasyon (Autocorrelation):** Durbin-Watson değeri ($1.779$), $1.5 - 2.5$ aralığında olduğu için artıklarda otokorelasyon sorunu bulunmamaktadır.

**Genel Sonuç:** Modelin tüm temel OLS varsayımları eksiksiz sağlanmıştır. Şikayet başına düşen şiddet mağduru oranını açıklamada okul terk oranı ve işsizlik oranı anlamlı bir katkı sunmazken, **devletin ayırdığı fon miktarı (`Funding_EUR`)** temel ve anlamlı belirleyici olarak öne çıkmaktadır.